[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day8_solution.ipynb)

# Day 8 · 정답 — LLM 과 프롬프트

모델을 열어 보고 API 로 데이터를 처리한다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`live` 와 `lab` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 모델을 연다

In [ ]:
# 1) 오늘 쓸 것들을 불러온다
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
print('torch', torch.__version__)

In [ ]:
# 2) 모델을 받아 온다 — 계수 15억 개짜리 한 대
NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(NAME)
# 어텐션을 꺼내 보려면 attn_implementation='eager' 여야 한다
model = AutoModelForCausalLM.from_pretrained(NAME, attn_implementation='eager')
model.eval()
print('계수 %.1f억 개' % (sum(p.numel() for p in model.parameters()) / 1e8))

In [ ]:
# 3) 장치로 옮긴다 — cpu 면 느리지만 돌기는 한다
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('장치', device)

## 2. 토큰 — 모델이 실제로 받는 단위

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 1.** 아래 문장이 몇 토큰인지 세어 `n` 에 담는다.

In [ ]:
q = '오늘 설비 점검에서 이상이 발견되었다'
n = len(tok(q)['input_ids'])

print(n)
assert isinstance(n, int) and n > 0

## 3. 임베딩 — 토큰이 숫자 줄이 된다

In [ ]:
# 토큰 하나가 몇 칸짜리 숫자 줄인지 본다
E = model.get_input_embeddings().weight
print('어휘 %d개 · 한 토큰 %d칸' % (len(tok), E.shape[1]))
print('표는 %d줄 — 계산이 빠른 크기로 맞춰 두느라 어휘보다 조금 길다' % E.shape[0])

In [ ]:
# 낱말 하나의 숫자 줄을 꺼내는 함수
def vec(w):
    return E[tok(w, add_special_tokens=False)['input_ids'][0]]

print('king 의 숫자 줄', tuple(vec(' king').shape))

In [ ]:
# 낱말 쌍 목록을 주면 가까운 정도를 재서 찍어 주는 함수
def show_sim(pairs):
    for a, b in pairs:
        v = F.cosine_similarity(vec(a), vec(b), dim=0).item()
        print('%-8s ~ %-8s  %.3f' % (a, b, v))

show_sim([(' king', ' queen'), (' king', ' banana')])

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 2.** 아래 목록에 **우리말 낱말 쌍 세 개**를 넣는다. 앞에 빈칸을 붙인다.
> 뜻이 가까운 쌍과 먼 쌍이 숫자로 갈리는지 본다.

In [ ]:
PAIRS = [(' 왕', ' 여왕'), (' 왕', ' 바나나'), (' 고양이', ' 개')]
show_sim(PAIRS)
print('우리말은 영어보다 덜 갈릴 수 있다 — 토큰이 쪼개져서다')

## 4. 다음 한 토큰

In [ ]:
# 앞부분을 넣고 다음 토큰의 점수를 받아 온다
head = '대한민국의 수도는'
x = tok(head, return_tensors='pt').to(device)
with torch.no_grad():
    logits = model(**x).logits[0, -1]
probs = logits.softmax(-1)
print('후보 %d개에 확률이 매겨졌다' % probs.shape[0])

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 3.** 1등 토큰의 확률을 `p1` 에 담는다.

In [ ]:
p1 = probs.max().item()

print('%.4f' % p1)
assert 0 < p1 <= 1

## 5. 온도 — 차이를 얼마나 벌릴지

In [ ]:
# 온도 목록을 주면 1등 확률을 재서 찍어 주는 함수
def show_temp(temps):
    for T in temps:
        p1 = (logits / T).softmax(-1).max().item()
        print('T=%.1f   1등 확률 %.4f' % (T, p1))

show_temp([1.0])

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 4.** 아래 목록에 **온도를 다섯 개** 넣는다. 0.1 부터 3.0 사이로 고른다.
> 낮추면 1에 붙고 높이면 낮아지는 것이 숫자로 보여야 한다.

In [ ]:
TEMPS = [0.1, 0.5, 1.0, 2.0, 3.0]
show_temp(TEMPS)
print('온도는 순위를 바꾸지 않는다 — 차이를 얼마나 크게 볼지만 정한다')

## 6. 어텐션 — 어디를 보고 고르나

In [ ]:
# 어텐션 가중치를 같이 받아 온다
s2 = 'The bank of the river was very steep'
x2 = tok(s2, return_tensors='pt').to(device)
with torch.no_grad():
    out = model(**x2, output_attentions=True)
A = out.attentions
print('층 %d개 · 층마다 머리 %d개 · 토큰 %d개' % (len(A), A[0].shape[1], A[0].shape[-1]))

In [ ]:
# 층과 머리 번호를 주면 그려 주는 함수
def draw(L, H):
    lab = [tok.decode([i]) for i in x2['input_ids'][0]]
    plt.figure(figsize=(5, 4))
    plt.imshow(A[L][0, H].float().cpu(), cmap='Purples')
    plt.xticks(range(len(lab)), lab, rotation=45, ha='right')
    plt.yticks(range(len(lab)), lab)
    plt.title('%d층 %d번 머리' % (L, H)); plt.show()

draw(14, 0)

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 5.** 층과 머리 번호를 바꿔 세 번 그려 본다. 층은 0~27, 머리는 0~11 이다.
> 앞 층과 뒤 층이 보는 자리가 다른지 확인한다.

In [ ]:
draw(0, 0)
draw(14, 0)
draw(26, 3)
print('머리마다 보는 자리가 다르다')

## 7. 프롬프트는 앞부분이다

In [ ]:
# 앞으로 쓸 답 만들기 함수 — 채팅 틀을 씌워 이어 쓰게 한다
def gen(user, system=None, n=56):
    ms = ([{'role': 'system', 'content': system}] if system else []) \
         + [{'role': 'user', 'content': user}]
    p = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True)
    x = tok(p, return_tensors='pt').to(device)
    with torch.no_grad():
        y = model.generate(**x, max_new_tokens=n, do_sample=False)
    return tok.decode(y[0][x['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
# 오늘 물어볼 한 문장
ask = '설비 점검에서 소음이 크고 진동이 있다.'
print(ask)

In [ ]:
# 낱말 목록을 주면 하나씩 물어봐 주는 함수
def ask_all(words):
    for w in words:
        print('[%s] %s' % (w, gen('%s가 뭐야? 한 문장으로 답해라.' % w, n=40)))
        print()

ask_all(['고로'])

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 6.** 칸 이름을 하나 더 넣어(`재발 방지:`) 다시 돌려 `out` 에 담는다.

In [ ]:
form2 = ask + '\n아래 형식으로만 답하라.\n증상: \n의심 원인: \n조치: \n재발 방지: '
out = gen(form2, n=120)

print(out)
assert '재발' in out

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 7.** 아래 목록에 **현장에서 쓰는 이름 세 개**를 넣고 답이 맞는지 본다.
> 틀린 답을 얼마나 자신 있게 말하는지 함께 확인한다.

In [ ]:
WORDS = ['고로', '전로', '소성로']
ask_all(WORDS)
print('모르면 비운 채 두지 않고 그럴듯한 말을 채워 넣는다')

## 8. 큰 모델로 데이터 처리하기

In [ ]:
# 키는 화면에 안 찍히게 받는다
import getpass, json, urllib.request, urllib.error
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')
print('키 길이', len(KEY))

In [ ]:
# 모델 하나에 물어보는 함수 — 실패해도 노트북이 멈추지 않게 한다
URL = 'https://integrate.api.nvidia.com/v1/chat/completions'

def nv(model, prompt, n=400, temp=0):
    body = json.dumps({'model': model, 'max_tokens': n, 'temperature': temp,
                       'messages': [{'role': 'user', 'content': prompt}]}).encode()
    req = urllib.request.Request(URL, data=body, headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=120) as f:
            return json.load(f)['choices'][0]['message']['content'].strip()
    except Exception as e:
        return '[실패] %s' % str(e)[:60]

print(nv('meta/llama-3.1-8b-instruct', '한 단어로만 답하라. 대한민국의 수도는?', 10))

In [ ]:
# 오늘 쓸 모델 — 크기가 다르다
MODELS = [('3B',  'meta/llama-3.2-3b-instruct'),
          ('8B',  'meta/llama-3.1-8b-instruct'),
          ('49B', 'nvidia/llama-3.3-nemotron-super-49b-v1')]
for tag, name in MODELS:
    print('%-4s %s' % (tag, name))

In [ ]:
# 손으로 적은 점검 기록 여섯 줄
LOG = '\n'.join([
 '3/4 09:12 A라인 3호기 소음 커짐, 베어링 교체 요청 - 김철수',
 '3/4 14:30 B라인 컨베이어 벨트 장력 느슨함. 조정함 - 이영희',
 '3/5 08:05 A라인 3호기 베어링 교체 완료, 소음 정상 - 김철수',
 '3/5 11:20 C라인 온도 센서 값 튐. 케이블 접촉 불량으로 확인, 재결선 - 박민수',
 '3/6 16:45 B라인 벨트 다시 느슨해짐. 장력 조정만으로는 안 될 듯, 교체 검토 필요 - 이영희',
 '3/7 10:00 정기 점검. 특이사항 없음 - 박민수'])
print(LOG)

In [ ]:
# 프롬프트 하나를 여러 크기에 넣어 나란히 찍어 주는 함수
def compare(prompt, n=500):
    for tag, name in MODELS:
        print('=' * 8, tag, '=' * 8)
        print(nv(name, prompt, n))
        print()

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 8.** 기준 한 줄만 고쳐서 `재결선` 이 완료로 나오게 만든다.

In [ ]:
RULE2 = '원인을 찾아 손을 댔으면 완료다. 부품 교체나 재점검이 남았으면 미완이다.'
T2 = TABLE.replace('조치가 끝났으면 완료, 후속 작업이 남았으면 미완이다.', RULE2)

print(nv('meta/llama-3.1-8b-instruct', T2, 700))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 9.** 아래 `___` 자리를 **자기 업무 데이터와 기준**으로 채워 표가 나오게 만든다.
> 한 번에 안 되면 **기준과 형식만** 고쳐 가며 세 번까지 해 본다.

In [ ]:
MY_DATA = '\n'.join([
 '어제 받았는데 화면에 금이 가 있어요',
 '주문한 지 일주일인데 아직 안 왔어요',
 '색이 사진이랑 너무 달라서 반품하고 싶어요'])
ROLE   = '너는 고객 문의를 분류하는 담당자다.'
RULE   = '환불 · 배송 · 제품하자 · 기타 넷 중 하나로 분류한다.'
FORMAT = '마크다운 표로만 답하라. 열은 문의 · 분류 · 근거 세 개다.'
MY_PROMPT = ('# 역할\n' + ROLE + '\n# 기준\n' + RULE + '\n'
             '# 입력\n' + MY_DATA + '\n# 형식\n' + FORMAT)
print(nv('meta/llama-3.1-8b-instruct', MY_PROMPT, 500))
print('한 번에 하나씩만 고친다 — 여러 곳을 바꾸면 무엇이 들었는지 모른다')

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 10.** 위에서 만든 `MY_PROMPT` 를 **크기가 다른 모델들에** 넣어 결과를 견준다.
> 크기를 키워야 되는 일인지, 프롬프트를 고쳐야 되는 일인지 가른다.

In [ ]:
compare(MY_PROMPT)
print('크기로 풀리는 것과 프롬프트로 풀리는 것은 다르다')